In [1]:
# ============================================================
# PHASE 1 — SUBJECT-LEVEL SPLIT FIX
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
AUDIT_DIR = os.path.join(PROJECT_DIR, 'audit')
os.makedirs(SPLITS_DIR, exist_ok=True)

# ============================================================
# Load verified subject mapping from Phase 0
# ============================================================
with open(os.path.join(AUDIT_DIR, 'subject_mapping.json')) as f:
    mapping = json.load(f)

record_to_subject = mapping['record_to_subject']
print(f"✅ Loaded subject mapping")
print(f"   Records: {len(record_to_subject)}")
print(f"   Subjects: {len(set(record_to_subject.values()))}")

# ============================================================
# Split SUBJECTS (not records)
# ============================================================
all_subjects = sorted(set(record_to_subject.values()))  # 47 subjects
print(f"\nTotal unique subjects: {len(all_subjects)}")

# Deterministic shuffle
rng = np.random.default_rng(42)
shuffled_subjects = all_subjects.copy()
rng.shuffle(shuffled_subjects)

# 80/10/10 split
n = len(shuffled_subjects)  # 47
n_train = int(0.80 * n)   # 37
n_val = int(0.10 * n)     # 4
n_test = n - n_train - n_val  # 6

train_subjects = set(shuffled_subjects[:n_train])
val_subjects = set(shuffled_subjects[n_train:n_train + n_val])
test_subjects = set(shuffled_subjects[n_train + n_val:])

print(f"\nSubject split:")
print(f"  Train: {len(train_subjects)} subjects")
print(f"  Val:   {len(val_subjects)} subjects")
print(f"  Test:  {len(test_subjects)} subjects")

# ============================================================
# Get records per split
# ============================================================
train_records_v2 = sorted([r for r, s in record_to_subject.items() if s in train_subjects])
val_records_v2 = sorted([r for r, s in record_to_subject.items() if s in val_subjects])
test_records_v2 = sorted([r for r, s in record_to_subject.items() if s in test_subjects])

print(f"\nRecord split:")
print(f"  Train: {len(train_records_v2)} records")
print(f"  Val:   {len(val_records_v2)} records")
print(f"  Test:  {len(test_records_v2)} records")

# ============================================================
# CRITICAL: Verify S025 (records 201, 202) in SAME split
# ============================================================
s025_records = ['201', '202']
s025_splits = set()
for r in s025_records:
    if r in train_records_v2: s025_splits.add('train')
    if r in val_records_v2: s025_splits.add('val')
    if r in test_records_v2: s025_splits.add('test')

assert len(s025_splits) == 1, f"S025 split across: {s025_splits}"
s025_split = list(s025_splits)[0]
print(f"\n✅ S025 (records 201, 202) → {s025_split.upper()}")

# ============================================================
# Leakage assertions
# ============================================================
assert set(train_subjects).isdisjoint(val_subjects), "Train ∩ Val not empty"
assert set(train_subjects).isdisjoint(test_subjects), "Train ∩ Test not empty"
assert set(val_subjects).isdisjoint(test_subjects), "Val ∩ Test not empty"
print("✅ NO SUBJECT LEAKAGE")

assert set(train_records_v2).isdisjoint(val_records_v2)
assert set(train_records_v2).isdisjoint(test_records_v2)
assert set(val_records_v2).isdisjoint(test_records_v2)
print("✅ NO RECORD LEAKAGE")

# ============================================================
# Save new split manifest
# ============================================================
split_v2 = {
    'protocol': 'STRICT_SUBJECT_DISJOINT',
    'random_seed': 42,
    'split_strategy': 'subject-level 80/10/10',
    'train_subjects': sorted(train_subjects),
    'val_subjects': sorted(val_subjects),
    'test_subjects': sorted(test_subjects),
    'train_records': train_records_v2,
    'val_records': val_records_v2,
    'test_records': test_records_v2,
    'total_subjects': len(all_subjects),
    'total_records': len(record_to_subject),
    's025_in_split': s025_split,
    'verified_no_leakage': True
}

split_path_v2 = os.path.join(SPLITS_DIR, 'mitbih_subject_split_v2.json')
with open(split_path_v2, 'w') as f:
    json.dump(split_v2, f, indent=2)

print(f"\n✅ New split saved: {split_path_v2}")

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 70)
print("PHASE 1 COMPLETE — SUBJECT-LEVEL SPLIT")
print("=" * 70)
print(f"\nTrain records ({len(train_records_v2)}): {train_records_v2}")
print(f"Val records   ({len(val_records_v2)}): {val_records_v2}")
print(f"Test records  ({len(test_records_v2)}): {test_records_v2}")
print(f"\n✅ Zero subject leakage")
print(f"✅ Zero record leakage")
print(f"✅ S025 (201, 202) in {s025_split.upper()}")
print("\n" + "=" * 70)
print("NEXT: Phase 2 — Label Definition")
print("=" * 70)

Mounted at /content/drive
✅ Loaded subject mapping
   Records: 48
   Subjects: 47

Total unique subjects: 47

Subject split:
  Train: 37 subjects
  Val:   4 subjects
  Test:  6 subjects

Record split:
  Train: 38 records
  Val:   4 records
  Test:  6 records

✅ S025 (records 201, 202) → TRAIN
✅ NO SUBJECT LEAKAGE
✅ NO RECORD LEAKAGE

✅ New split saved: /content/drive/MyDrive/ecg-transcovnet/splits/mitbih_subject_split_v2.json

PHASE 1 COMPLETE — SUBJECT-LEVEL SPLIT

Train records (38): ['103', '104', '105', '106', '107', '109', '111', '112', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '215', '217', '220', '221', '222', '223', '228', '230', '231', '232', '234']
Val records   (4): ['100', '113', '115', '233']
Test records  (6): ['101', '102', '108', '114', '214', '219']

✅ Zero subject leakage
✅ Zero record leakage
✅ S025 (201, 202) in TRAIN

NEXT: Phase 2 — Label Definition
